## Importaciones

In [ ]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np

## Conexión a MongoDB

In [ ]:
client = MongoClient("mongodb://localhost:27017/")
db = client.traffic_system

# Verificar conexión
print("Colecciones disponibles:", db.list_collection_names())

## Consultar videos procesados

In [ ]:
videos = list(db.videos.find())
print(f"Total de videos procesados: {len(videos)}")

# Mostrar información básica de los videos
for video in videos:
    print(f"\nVideo ID: {video.get('_id')}")
    print(f"Nombre: {video.get('filename')}")
    print(f"Fecha: {video.get('created_at')}")
    print(f"Estado: {video.get('status')}")

#

In [ ]:
detections = list(db.detections.find())
print(f"Total de detecciones: {len(detections)}")

# Convertir a DataFrame
df_detections = pd.DataFrame(detections)
df_detections.head()

## Analisis temporal

In [ ]:
# Convertir timestamps a datetime
df_detections['timestamp'] = pd.to_datetime(df_detections['timestamp'])
df_detections['hour'] = df_detections['timestamp'].dt.hour
df_detections['date'] = df_detections['timestamp'].dt.date

# Gráfico de detecciones por hora
plt.figure(figsize=(12, 6))
sns.countplot(data=df_detections, x='hour')
plt.title('Detecciones por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Número de Detecciones')
plt.show()

## Analisis por video

In [ ]:
# Agrupar detecciones por video
video_stats = df_detections.groupby('video_id').agg({
    'timestamp': ['count', 'min', 'max'],
    'detections': lambda x: sum(len(d) for d in x)
}).reset_index()

video_stats.columns = ['video_id', 'total_frames', 'inicio', 'fin', 'total_detecciones']
print("\nEstadísticas por video:")
print(video_stats)

## Análisis de confianza

In [ ]:
confidences = []
for det in df_detections['detections']:
    for d in det:
        confidences.append(d['confidence'])

plt.figure(figsize=(10, 6))
plt.hist(confidences, bins=20)
plt.title('Distribución de Niveles de Confianza')
plt.xlabel('Confianza')
plt.ylabel('Frecuencia')
plt.show()

## Exportar resultados

In [ ]:
# Guardar estadísticas en CSV
video_stats.to_csv('estadisticas_videos.csv', index=False)

# Guardar detecciones procesadas
df_detections.to_csv('detecciones_procesadas.csv', index=False)

print("Archivos exportados correctamente")